<a href="https://colab.research.google.com/github/smerarawal/Smart-Hospital-Appointment-System-with-No-Show-Predictor-and-Reschedule-Generation/blob/main/dbms_final_last.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install flask mysql-connector-python scikit-learn xgboost numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 34.9 MB/s eta 0:00:00


In [ ]:
"""
Flask Backend — MedFlow No-Show Prediction System
==================================================
WHERE: Run on your laptop
HOW:   pip install flask mysql-connector-python scikit-learn xgboost
       python app.py
THEN:  Open http://localhost:5000 in browser

PUT rf_model.pkl in the same folder as this file.
"""

from flask import Flask, jsonify, request, render_template, send_from_directory
import mysql.connector
import numpy as np
import pickle
import os
import json

app = Flask(__name__)

# ── DB Config ─────────────────────────────────────────────────────────────────
DB_CONFIG = {
    'host':     'centerbeam.proxy.rlwy.net',
    'user':     'root',
    'password': 'uqqTtnLRCTBzVhUNDorIPByIknjemOAG',          # ← YOUR Railway password
    'database': 'hospital_db',
    'port':     46001
}

# ── Load ML model ─────────────────────────────────────────────────────────────
MODEL_PATH = os.path.join(os.path.dirname(__file__), 'rf_model.pkl')
try:
    rf_model = pickle.load(open(MODEL_PATH, 'rb'))
    print("✅ RF model loaded")
except FileNotFoundError:
    rf_model = None
    print("⚠️  rf_model.pkl not found — put it in the same folder as app.py")

DEPARTMENTS = ['Cardiology','Dermatology','General','Oncology','Orthopaedics','Psychiatry']
DAYS        = ['Monday','Tuesday','Wednesday','Thursday','Friday']

def get_db():
    return mysql.connector.connect(**DB_CONFIG)

def encode_patient(age, lead_time, past_noshow, past_cancel, hour, dept, day):
    f = [age, lead_time, past_noshow, past_cancel, hour]
    for d in DEPARTMENTS: f.append(1 if dept==d else 0)
    for d in DAYS:        f.append(1 if day==d else 0)
    return np.array(f).reshape(1,-1)

# ── ROUTES ────────────────────────────────────────────────────────────────────

@app.route('/')
def index():
    return send_from_directory('.', 'index.html')

@app.route('/static/<path:filename>')
def static_files(filename):
    return send_from_directory('static', filename)


# ── API: Dashboard stats ──────────────────────────────────────────────────────
@app.route('/api/dashboard')
def dashboard():
    try:
        conn = get_db(); cursor = conn.cursor(dictionary=True)

        cursor.execute("SELECT COUNT(*) AS total FROM synthetic_patients")
        total = cursor.fetchone()['total']

        cursor.execute("SELECT COUNT(*) AS high FROM synthetic_predictions WHERE risk_label='High'")
        high = cursor.fetchone()['high']

        cursor.execute("SELECT COUNT(*) AS medium FROM synthetic_predictions WHERE risk_label='Medium'")
        medium = cursor.fetchone()['medium']

        cursor.execute("SELECT COUNT(*) AS low FROM synthetic_predictions WHERE risk_label='Low'")
        low = cursor.fetchone()['low']

        cursor.execute("""
            SELECT SUM(correct='1') AS correct, COUNT(*) AS total
            FROM synthetic_predictions
        """)
        acc = cursor.fetchone()
        accuracy = round(acc['correct'] / acc['total'] * 100, 1) if acc['total'] else 0

        cursor.execute("""
            SELECT department,
                   ROUND(AVG(no_show)*100,1) AS noshow_pct
            FROM synthetic_patients
            GROUP BY department ORDER BY noshow_pct DESC
        """)
        by_dept = cursor.fetchall()

        cursor.execute("""
            SELECT day_of_week,
                   ROUND(AVG(no_show)*100,1) AS noshow_pct
            FROM synthetic_patients
            GROUP BY day_of_week ORDER BY noshow_pct DESC
        """)
        by_day = cursor.fetchall()

        cursor.close(); conn.close()
        return jsonify({
            'total': total, 'high': high, 'medium': medium,
            'low': low, 'accuracy': accuracy,
            'by_dept': by_dept, 'by_day': by_day
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── API: Predict single patient ───────────────────────────────────────────────
@app.route('/api/predict', methods=['POST'])
def predict():
    if rf_model is None:
        return jsonify({'error': 'Model not loaded'}), 500
    try:
        data = request.json
        features = encode_patient(
            data['age'], data['lead_time_days'],
            data['past_noshow_count'], data['past_cancellations'],
            data['hour_of_day'], data['department'], data['day_of_week']
        )
        prob = float(rf_model.predict_proba(features)[0][1])
        risk = 'High' if prob>=0.6 else ('Medium' if prob>=0.35 else 'Low')

        if prob >= 0.6:
            rec = "High no-show risk. Recommend controlled overbooking. Send SMS reminder 24h before."
        elif prob >= 0.35:
            rec = "Moderate risk. Consider overbooking if another high-risk patient shares this slot."
        else:
            rec = "Low no-show risk. No overbooking needed. Standard reminder protocol applies."

        return jsonify({'probability': round(prob, 3), 'risk': risk, 'recommendation': rec})
    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── API: All 100 synthetic predictions ───────────────────────────────────────
@app.route('/api/predictions')
def predictions():
    try:
        conn = get_db(); cursor = conn.cursor(dictionary=True)
        cursor.execute("""
            SELECT
                sp.patient_id, sp.age, sp.department,
                sp.day_of_week, sp.past_noshow_count,
                sp.lead_time_days, sp.no_show AS actual,
                sq.noshow_probability, sq.risk_label,
                sq.model_used, sq.correct
            FROM synthetic_patients sp
            JOIN synthetic_predictions sq ON sp.patient_id = sq.patient_id
            ORDER BY sq.noshow_probability DESC
        """)
        rows = cursor.fetchall()
        cursor.close(); conn.close()
        return jsonify(rows)
    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── API: Overbooking decision ─────────────────────────────────────────────────
@app.route('/api/overbook', methods=['POST'])
def overbook():
    data = request.json
    p1 = float(data['p1'])
    p2 = float(data['p2'])
    capacity = int(data.get('capacity', 1))
    threshold = float(data.get('threshold', 0.60))

    show1 = round(1-p1, 3)
    show2 = round(1-p2, 3)
    E = round(show1+show2, 3)
    both_high = p1>=threshold and p2>=threshold
    safe = E<=capacity
    decision = "ALLOW OVERBOOKING" if both_high and safe else "DENY"

    return jsonify({
        'p1': p1, 'p2': p2,
        'show1': show1, 'show2': show2,
        'expected_attendance': E,
        'both_high_risk': both_high,
        'safe': safe,
        'decision': decision
    })


# ── API: Reappointment recommendations ───────────────────────────────────────
@app.route('/api/reappoint/<int:patient_id>')
def reappoint(patient_id):
    if rf_model is None:
        return jsonify({'error': 'Model not loaded'}), 500
    try:
        conn = get_db(); cursor = conn.cursor(dictionary=True)
        cursor.execute(f"""
            SELECT age, past_noshow_count, past_cancellations,
                   department, day_of_week, hour_of_day
            FROM synthetic_patients
            WHERE patient_id={patient_id} LIMIT 1
        """)
        patient = cursor.fetchone()
        cursor.close(); conn.close()

        if not patient:
            return jsonify({'error': 'Patient not found'}), 404

        candidates = []
        for day in DAYS:
            for hour in [8,9,10,11,13,14,15]:
                for lead in [3,7,14,21]:
                    features = encode_patient(
                        patient['age'], lead,
                        patient['past_noshow_count'],
                        patient['past_cancellations'],
                        hour, patient['department'], day
                    )
                    prob = float(rf_model.predict_proba(features)[0][1])
                    candidates.append({
                        'day': day, 'time': f"{hour:02d}:00",
                        'lead_days': lead,
                        'noshow_prob': round(prob, 3),
                        'show_prob': round(1-prob, 3)
                    })

        candidates.sort(key=lambda x: x['noshow_prob'])
        return jsonify({
            'patient': patient,
            'recommendations': candidates[:5]
        })
    except Exception as e:
        return jsonify({'error': str(e)}), 500


# ── API: SQL Analysis queries ─────────────────────────────────────────────────
@app.route('/api/analysis/<query_name>')
def analysis(query_name):
    queries = {
        'by_dept': """
            SELECT department,
                   COUNT(*) AS total,
                   SUM(no_show) AS noshows,
                   ROUND(AVG(no_show)*100,1) AS noshow_pct
            FROM synthetic_patients
            GROUP BY department ORDER BY noshow_pct DESC
        """,
        'by_day': """
            SELECT day_of_week,
                   COUNT(*) AS total,
                   ROUND(AVG(no_show)*100,1) AS noshow_pct
            FROM synthetic_patients
            GROUP BY day_of_week ORDER BY noshow_pct DESC
        """,
        'risk_dist': """
            SELECT risk_label,
                   COUNT(*) AS total,
                   ROUND(AVG(noshow_probability)*100,1) AS avg_prob_pct,
                   ROUND(MIN(noshow_probability),3) AS min_prob,
                   ROUND(MAX(noshow_probability),3) AS max_prob
            FROM synthetic_predictions
            GROUP BY risk_label ORDER BY avg_prob_pct DESC
        """,
        'accuracy': """
            SELECT
                COUNT(*) AS total,
                SUM(correct='1') AS correct,
                ROUND(SUM(correct='1')*100.0/COUNT(*),1) AS accuracy_pct
            FROM synthetic_predictions
        """,
        'high_risk': """
            SELECT sp.patient_id, sp.age, sp.department,
                   sp.past_noshow_count, sp.lead_time_days,
                   sq.noshow_probability, sq.risk_label
            FROM synthetic_patients sp
            JOIN synthetic_predictions sq ON sp.patient_id = sq.patient_id
            WHERE sq.risk_label = 'High'
            ORDER BY sq.noshow_probability DESC
        """
    }
    if query_name not in queries:
        return jsonify({'error': 'Unknown query'}), 400
    try:
        conn = get_db(); cursor = conn.cursor(dictionary=True)
        cursor.execute(queries[query_name])
        rows = cursor.fetchall()
        cursor.close(); conn.close()
        return jsonify(rows)
    except Exception as e:
        return jsonify({'error': str(e)}), 500


if __name__ == '__main__':
    print("\n🏥 MedFlow Intelligence Platform")
    print("   Running on http://localhost:5000\n")
    app.run(debug=True, port=5000)

NameError: name '__file__' is not defined

In [ ]:
!pip install pymysql

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.3/45.3 kB 3.7 MB/s eta 0:00:00


In [ ]:
# 3. Predict
import pickle

# --> THIS IS THE MAGIC LINE WE WERE MISSING <--
rf_model = pickle.load(open('rf_model.pkl', 'rb'))

def encode_patient(row):
    f = [row['age'], row['lead_time_days'], row['past_noshow_count'], row['past_cancellations'], row['hour_of_day']]
    for d in departments: f.append(1 if row['department'] == d else 0)
    for d in days: f.append(1 if row['day_of_week'] == d else 0)
    return np.array(f).reshape(1,-1)

predictions = []
for _, row in new_patients.iterrows():
    prob = rf_model.predict_proba(encode_patient(row))[0][1]
    risk = 'High' if prob >= 0.6 else ('Medium' if prob >= 0.35 else 'Low')
    predictions.append({
        'patient_id': int(row['patient_id']),
        'noshow_probability': round(prob, 4),
        'risk_label': risk,
        'model_used': 'Random Forest',
        'correct': 1 if (prob >= 0.5) == bool(row['no_show']) else 0
    })

predictions_df = pd.DataFrame(predictions)

In [ ]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# 1. Connect to Railway
DB_HOST="centerbeam.proxy.rlwy.net"
DB_USER="root"
DB_PASSWORD="uqqTtnLRCTBzVhUNDorIPByIknjemOAG" # Use your actual password
DB_PORT=46001
DB_NAME="hospital_db"
engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}")

# 2. Generate 100 Patients
np.random.seed(99)
departments = ['Cardiology', 'Dermatology', 'General', 'Oncology', 'Orthopaedics', 'Psychiatry']
days = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

new_patients = pd.DataFrame({
    'patient_id': range(2000, 2100),
    'age': np.random.randint(18, 85, 100),
    'department': np.random.choice(departments, 100),
    'day_of_week': np.random.choice(days, 100),
    'hour_of_day': np.random.choice([8,9,10,11,13,14,15,16], 100),
    'lead_time_days': np.random.randint(1, 90, 100),
    'past_noshow_count': np.random.randint(0, 10, 100),
    'past_cancellations': np.random.randint(0, 6, 100),
    'no_show': np.random.choice([0, 1], 100, p=[0.75, 0.25])
})

# 3. Predict
def encode_patient(row):
    f = [row['age'], row['lead_time_days'], row['past_noshow_count'], row['past_cancellations'], row['hour_of_day']]
    for d in departments: f.append(1 if row['department'] == d else 0)
    for d in days: f.append(1 if row['day_of_week'] == d else 0)
    return np.array(f).reshape(1,-1)

predictions = []
for _, row in new_patients.iterrows():
    prob = rf_model.predict_proba(encode_patient(row))[0][1]
    risk = 'High' if prob >= 0.6 else ('Medium' if prob >= 0.35 else 'Low')
    predictions.append({
        'patient_id': int(row['patient_id']),
        'noshow_probability': round(prob, 4),
        'risk_label': risk,
        'model_used': 'Random Forest',
        'correct': 1 if (prob >= 0.5) == bool(row['no_show']) else 0
    })

predictions_df = pd.DataFrame(predictions)

# 4. Push to the tables the Dashboard is expecting!
new_patients.to_sql('synthetic_patients', engine, if_exists='replace', index=False)
predictions_df.to_sql('synthetic_predictions', engine, if_exists='replace', index=False)

print("✅ Dashboard Data Generated Successfully!")

✅ Dashboard Data Generated Successfully!
